# 00 - Setup: carga dos dados no SQL Server

Cria o database **LojaDB** no SQL Server 2025 e carrega as tabelas `clientes` e `pedidos`
a partir dos CSVs da pasta `data/` (a mesma loja fictícia do trabalho 1).

**Pré-requisitos:** `docker compose up -d` executado, arquivo `.env` criado e ODBC Driver 18 instalado.

## 1. Configuração e conexão

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
if not (RAIZ / '.env').exists():
    raise FileNotFoundError('Arquivo .env não encontrado. Rode: cp .env.example .env')
load_dotenv(RAIZ / '.env', override=True)

import pandas as pd
import pyodbc

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')
DATA_DIR    = RAIZ / 'data'

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

Servidor: localhost:1433
Database: LojaDB


In [2]:
# Conexão ao SQL Server (master) para criar o database
conn_master = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao SQL Server (master) com sucesso!')

Conectado ao SQL Server (master) com sucesso!


## 2. Criar o database LojaDB

In [3]:
cursor_master.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_DATABASE}')
    BEGIN
        CREATE DATABASE [{DB_DATABASE}]
    END
""")
print(f'Database [{DB_DATABASE}] criado/verificado com sucesso!')
cursor_master.close()
conn_master.close()

Database [LojaDB] criado/verificado com sucesso!


In [4]:
# Conectar ao database LojaDB
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso!')

Conectado ao [LojaDB] com sucesso!


## 3. Criar as tabelas (DDL)

Relação **1 : N** — um cliente possui vários pedidos (`pedidos.id_cliente` é chave estrangeira de `clientes`).

In [5]:
ddl = {
    'clientes': """
        IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'clientes')
        CREATE TABLE clientes (
            id_cliente INT PRIMARY KEY,
            nome       NVARCHAR(100) NOT NULL,
            cidade     NVARCHAR(100),
            uf         CHAR(2)
        )
    """,
    'pedidos': """
        IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'pedidos')
        CREATE TABLE pedidos (
            id_pedido      INT PRIMARY KEY,
            id_cliente     INT NOT NULL,
            produto        NVARCHAR(100) NOT NULL,
            quantidade     INT NOT NULL,
            valor_unitario DECIMAL(10,2) NOT NULL,
            data_pedido    DATE NOT NULL,
            status         NVARCHAR(20) NOT NULL,
            CONSTRAINT fk_pedidos_clientes
                FOREIGN KEY (id_cliente) REFERENCES clientes (id_cliente)
        )
    """,
}

for nome, sql in ddl.items():
    cursor.execute(sql)
    print(f'Tabela [{nome}] criada/verificada')

Tabela [clientes] criada/verificada
Tabela [pedidos] criada/verificada


## 4. Carregar os dados dos CSVs

In [6]:
# Ordem de carga respeita a chave estrangeira: clientes antes de pedidos
tabelas = ['clientes', 'pedidos']

for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    existentes = cursor.fetchone()[0]
    if existentes > 0:
        print(f'  {tabela}: já contém {existentes} registros, pulando...')
        continue

    df = pd.read_csv(DATA_DIR / f'{tabela}.csv')

    # Remover espaços em colunas de texto
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.strip()

    # Converter NaN para None (NULL no SQL Server)
    df = df.astype(object).where(df.notna(), None)

    cols = ', '.join(df.columns)
    placeholders = ', '.join('?' for _ in df.columns)
    insert_sql = f'INSERT INTO {tabela} ({cols}) VALUES ({placeholders})'
    dados = [tuple(linha) for linha in df.itertuples(index=False, name=None)]

    cursor.executemany(insert_sql, dados)
    print(f'  {tabela}: {len(dados)} registros inseridos')

print('\nCarga de dados concluída!')

  clientes: 4 registros inseridos
  pedidos: 5 registros inseridos

Carga de dados concluída!


## 5. Validação

In [7]:
print(f'{"Tabela":<15} {"Registros":>10}')
print('-' * 27)
for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    print(f'{tabela:<15} {cursor.fetchone()[0]:>10}')

Tabela           Registros
---------------------------
clientes                 4
pedidos                  5


In [8]:
# Amostra dos dados (com JOIN, para mostrar a relação 1:N)
cursor.execute("""
    SELECT p.id_pedido, c.nome, p.produto, p.quantidade, p.valor_unitario, p.status
    FROM pedidos p
    JOIN clientes c ON c.id_cliente = p.id_cliente
    ORDER BY p.id_pedido
""")
colunas = [d[0] for d in cursor.description]
print(pd.DataFrame.from_records(cursor.fetchall(), columns=colunas).to_string(index=False))

 id_pedido       nome  produto  quantidade valor_unitario    status
       101  Ana Souza Notebook           1        4500.00      PAGO
       102 Bruno Lima    Mouse           2          80.00      PAGO
       103 Carla Dias  Monitor           1        1200.00  PENDENTE
       104  Ana Souza  Teclado           1         250.00  PENDENTE
       105 Diego Melo  Headset           1         300.00 CANCELADO


In [9]:
cursor.close()
conn.close()
print('Conexão encerrada.')

Conexão encerrada.
